[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Overfitting and Regularization &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import copy
import os
import warnings

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


The worked examples' tensors and functions, rebuilt here so that each answer runs on its own.
`fit_full_batch` also takes the stations to train on, which the fifth answer uses.

In [2]:
def as_tensors(rows):
    """The standardized features of these rows, and their January means as a column."""
    return torch.from_numpy(scaled[rows]), torch.from_numpy(jan_mean[rows]).unsqueeze(1)


few_rows = train_rows[:100]
X_few, y_few = as_tensors(few_rows)
X_valid, y_valid = as_tensors(valid_rows)
X_test, y_test = as_tensors(test_rows)
mse = torch.nn.MSELoss()


def large_network(dropout=0.0):
    """A 4-128-128-1 network, with dropout after each hidden layer when dropout is above 0."""
    layers = [torch.nn.Linear(4, 128), torch.nn.ReLU()]
    if dropout:
        layers.append(torch.nn.Dropout(dropout))
    layers += [torch.nn.Linear(128, 128), torch.nn.ReLU()]
    if dropout:
        layers.append(torch.nn.Dropout(dropout))
    layers.append(torch.nn.Linear(128, 1))
    return torch.nn.Sequential(*layers)


def fit_full_batch(model, optimizer, epochs=400, X=X_few, y=y_few):
    """One step per epoch on X and y; returns the training and validation loss of every epoch."""
    history = {"loss": [], "val_loss": []}
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        mse(model(X), y).backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            history["loss"].append(mse(model(X), y).item())
            history["val_loss"].append(mse(model(X_valid), y_valid).item())
    return history


def show(history, every=50):
    """Print both losses every few epochs, then the epoch with the lowest validation loss."""
    for epoch in range(every, len(history["loss"]) + 1, every):
        print(f"epoch {epoch:3}: training loss {history['loss'][epoch - 1]:.3f}, "
              f"validation loss {history['val_loss'][epoch - 1]:.3f}")
    best = int(np.argmin(history["val_loss"]))
    print(f"lowest validation loss at epoch {best + 1}: {history['val_loss'][best]:.3f}")

**1.** Train the large network with a dropout rate of 0.5, from `torch.manual_seed(0)`, with `fit_full_batch`, and print its losses with `show`.

In [3]:
torch.manual_seed(0)
half = large_network(dropout=0.5)
show(fit_full_batch(half, torch.optim.Adam(half.parameters(), lr=0.01)), every=100)

epoch 100: training loss 0.913, validation loss 1.057
epoch 200: training loss 0.701, validation loss 0.981
epoch 300: training loss 0.661, validation loss 0.937
epoch 400: training loss 0.671, validation loss 0.983
lowest validation loss at epoch 268: 0.830


A rate of 0.5 trains more slowly. The training loss is 0.671 at epoch 400, against 0.475 at a rate of 0.3, and the lowest validation loss is 0.83, at epoch 268, against 0.787.

**2.** Train the large network with `torch.optim.AdamW` at a weight decay of 0.1, and compare its lowest validation loss with the worked example's at a weight decay of 1.0.

In [4]:
torch.manual_seed(0)
light = large_network()
show(fit_full_batch(light, torch.optim.AdamW(light.parameters(), lr=0.01, weight_decay=0.1)), every=100)

epoch 100: training loss 0.605, validation loss 0.821
epoch 200: training loss 0.381, validation loss 0.929
epoch 300: training loss 0.276, validation loss 1.045
epoch 400: training loss 0.202, validation loss 1.095
lowest validation loss at epoch 97: 0.815


A decay of 0.1 ends at a validation loss of 1.095 by epoch 400, against 0.965 with a decay of 1.0, and its lowest, 0.815 at epoch 97, is below 1.0's 0.873. Both differences come from one seed, too little to rank the two settings.

**3.** Run the early-stopping loop with a patience of 10 epochs, and print the epoch it stopped at, the best epoch and the validation loss of the weights it loads back.

In [5]:
torch.manual_seed(0)
stopped = large_network()
optimizer = torch.optim.Adam(stopped.parameters(), lr=0.01)
best_loss, best_epoch, best_weights, epoch = float("inf"), 0, None, 0
while epoch - best_epoch < 10:
    epoch += 1
    stopped.train()
    optimizer.zero_grad()
    mse(stopped(X_few), y_few).backward()
    optimizer.step()
    stopped.eval()
    with torch.no_grad():
        val_loss = mse(stopped(X_valid), y_valid).item()
    if val_loss < best_loss:
        best_loss, best_epoch, best_weights = val_loss, epoch, copy.deepcopy(stopped.state_dict())

stopped.load_state_dict(best_weights)
with torch.no_grad():
    print(f"stopped after epoch {epoch}, best epoch {best_epoch}, "
          f"validation loss {mse(stopped(X_valid), y_valid).item():.3f}")

stopped after epoch 86, best epoch 76, validation loss 0.840


With a patience of 10, training stops after epoch 86, ten epochs after the same best epoch, 76, and loads back the same weights, 40 epochs sooner than a patience of 50.

**4.** Give both hidden layers of a Keras version of the large network `kernel_regularizer=keras.regularizers.L2(0.001)`, compile it with `metrics=["mse"]`, fit it with an `EarlyStopping` like the worked example's that watches `val_mse`, and print the best epoch and that epoch's `val_mse`.

In [6]:
keras.utils.set_random_seed(0)
penalized = keras.Sequential([
    keras.Input((4,)),
    keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.L2(0.001)),
    keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.L2(0.001)),
    keras.layers.Dense(1),
])
penalized.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse", metrics=["mse"])
early = keras.callbacks.EarlyStopping(monitor="val_mse", patience=50, restore_best_weights=True)
fitted = penalized.fit(scaled[few_rows], jan_mean[few_rows],
                       validation_data=(scaled[valid_rows], jan_mean[valid_rows]),
                       epochs=400, batch_size=100, verbose=0, callbacks=[early])

print("best epoch:", early.best_epoch + 1)
print(f"val_mse {fitted.history['val_mse'][early.best_epoch]:.3f} | "
      f"val_loss, with the penalty {fitted.history['val_loss'][early.best_epoch]:.3f}")

best epoch: 65
val_mse 0.813 | val_loss, with the penalty 0.956


With an L2 penalty of 0.001, the best epoch is 65, where the validation mean squared error is 0.813
and the validation loss, which includes the penalty, is 0.956. The plain Keras network's best in the
worked example, 0.803, is a mean squared error alone, so it compares with the 0.813. That is one
seed, too little to say whether the penalty helps.

**5.** Write a version of `fit_full_batch` that takes the training stations as arguments, train the large network without regularization on all 800 training stations, one full batch a step, for 400 epochs, and see whether its validation loss still turns.

In [7]:
X_train, y_train = as_tensors(train_rows)
torch.manual_seed(0)
everything = large_network()
show(fit_full_batch(everything, torch.optim.Adam(everything.parameters(), lr=0.01), X=X_train, y=y_train), every=100)

epoch 100: training loss 0.761, validation loss 0.778
epoch 200: training loss 0.732, validation loss 0.780
epoch 300: training loss 0.700, validation loss 0.789
epoch 400: training loss 0.676, validation loss 0.794
lowest validation loss at epoch 219: 0.776


With all 800 stations, the validation loss falls to 0.776 at epoch 219 and rises only to 0.794 by epoch 400, while the training loss is 0.676. Eight times the data leaves the same 17,281 weights much less room to fit noise.

---

&#8592; **Back to:** [Overfitting and Regularization](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/11-overfitting-and-regularization.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
